# Biweekly-window error scorecard

## Purpose

Compare hindcast skill across lead windows (week1-2, week3-4, week5-6, week7-8; day 1 = initial date) for atmosphere and land variables, CTRL10 vs CAPT10 vs DART40, in both seasons. Each heat-map cell holds four triangles, one per window: top week1-2, right week3-4, bottom week5-6, left week7-8.

## Inputs

Member output of the hindcasts in `util/bias_metrics.py` under `post/<atm|lnd>/180x360_aave/ts/`, read through `util.model_daily`, each variable sampled so model and reference describe the same time (see Notes): 00Z-24Z daily means from `atm/.../ts/daily_mean` (PSL, TREFHT, TS, 10-m wind speed, PRECT, FLUT, surface fluxes) and the ELM `lnd/.../ts/daily` series, and 00Z instantaneous states from `ts/daily_00z` (U, V, T, Q, Z on pressure levels, which exist only as once-daily 00Z output) verified against ERA5 at 00Z. References under `OBSERVATION_ROOT`: ERA5 (PSL; U, V, T, Q, Z at 200, 500, 700, 850, 925 hPa; TREFHT; skin temperature TS; 10-m wind speed; surface latent and sensible heat and net shortwave, downward shortwave, and downward longwave radiation; land: TREFHT for TSA, skin temperature TS for TG, LHFLX, SHFLX), GPM IMERG (PRECT), NOAA interpolated OLR (FLUT), ESA CCI soil moisture (top 5 cm). Land variables are scored over land; pressure-level variables only where the level is above ground (ERA5 surface-pressure climatology of the season's months above the level), which removes 850 and 925 hPa over Antarctica, Tibet, the Andes, and Greenland (and 925 hPa over other high terrain).

## Metrics

- **Anomaly PCC**: centered, cos-latitude weighted pattern correlation of window-mean anomalies; model and reference anomalies are both taken from the same reference climatology of those days.
- **MSSS**: mean-square skill score of the window-mean ensemble mean against the climatology forecast, `1 - (RMSE / RMSE_clim)^2`, where `RMSE_clim` is the RMSE of the reference climatology of those days. 1 = perfect, 0 = no better than climatology, negative = worse (blue = beats climatology). Every cell, CTRL included, is scored on the same absolute scale.
- **CRPSS**: skill of the whole ensemble, `1 - CRPS / CRPS_clim`. CRPS is the fair CRPS of the members' window means at each gridpoint (corrected for ensemble size, so 10- and 40-member ensembles compare fairly), area-averaged; the reference is the climatological ensemble, each climatology year's window means. It rewards an ensemble that is both accurate and has the right spread; mean bias counts in full.
- **Mean bias** (`REMOVE_MEAN_BIAS`): optionally remove the region's area-mean bias from both the model and the climatology forecast before the RMSE (centered RMSE; CRPS is unaffected). A uniform offset, such as the EAM-DART global surface-pressure loss in PSL and Z500, otherwise dominates the RMSE; the bias itself stays in the table (`bias`).

## Climatology

One shared period for every variable (`CLIM_YEARS`, default 2001-2017, the overlap of all reference records), leaving out the verified year (`CLIM_EXCLUDE_YEARS`). Daily references are averaged per calendar day and smoothed with a 31-day running mean; ERA5 TREFHT, TS, LHFLX, and SHFLX, archived daily only for 2011-2012, use the ERA5 monthly climatology interpolated to days.

## Outputs

Window-mean NetCDF caches and the score table under `DIAGNOSTIC_OUTPUT_ROOT/data/fcst_window_scorecard/`; heat maps under `DIAGNOSTIC_OUTPUT_ROOT/figure/fcst_window_scorecard/`. Review the user setup, then run the notebook from top to bottom.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

## 2. User setup

Paths; the seasons and experiments compared, variables, regions, climatology period, and run options. Edit this cell for normal changes.

In [2]:
INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/compyfs/zhan391/acme_init/Observations")  # observations (shared archive)
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")  # land mask
WORKFLOW_NAME = "fcst_window_scorecard"

# What to compare: season -> experiments (the CTRL run is the RMSE baseline).
CASES = {
    "S0": ("CTRL10-S0", "CAPT10-S0", "DART40-S0"),  # Jan-Feb 2012
    "S1": ("CTRL10-S1", "CAPT10-S1", "DART40-S1"),  # Jun-Jul 2012
}

# Variables processed and scored; each is cached separately, so adding one later
# only computes that one.
LEVELS = (200, 500, 700, 850, 925)  # hPa
UPPER_AIR = tuple(f"{name}{level}" for name in ("U", "V", "T", "Q", "Z") for level in LEVELS)
SURFACE = ("PSL", "TREFHT", "TS", "WSPD10", "PRECT", "FLUT", "LHFLX", "SHFLX", "FSNS", "FSDS", "FLDS")
LAND = ("TSA", "TG", "SOILM_5CM", "EFLX_LH_TOT", "FSH")
VARIABLES = SURFACE + UPPER_AIR + LAND

# Figures: one heat map per named set and region; any subset of VARIABLES.
PLOT_SETS = {
    "surface": SURFACE,
    "upper_air": UPPER_AIR,
    "land": LAND,
}
PLOT = ("surface", "upper_air", "land")  # sets drawn below
# global, land, ocean, Tropics, CONUS, NHMidLat, NINO3.4, MAR_CONT (one figure each)
REGIONS = ("global",)

CLIM_YEARS = (2001, 2017)  # shared climatology period for every variable
CLIM_EXCLUDE_YEARS = (2012,)  # leave the verified year out
MIN_DAYS = 10  # window days a gridpoint needs (of 14) where the reference has gaps
REMOVE_MEAN_BIAS = False  # True: score centered RMSE (region-mean bias removed)

MAX_MEMBERS = None  # e.g. 2 for a quick test; caches record the ensemble size
FORCE_COMPUTE = True
SHOW = True
FIGURE_DPI = 300

## 3. Driver: read, process, and save data

Reuse valid window-mean caches or build missing ones, then score every window and save the table.

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.forecast_ensemble_ts import REGIONS as AVAILABLE_REGIONS
from util.forecast_window_scorecard import build_window_means, window_scores

unknown = set(REGIONS) - set(AVAILABLE_REGIONS)
if unknown:
    raise ValueError(f"Unknown regions {sorted(unknown)}; choose from {list(AVAILABLE_REGIONS)}")
variables = VARIABLES
data_dir, figure_dir = workflow_output_dirs(WORKFLOW_NAME, output_root=DIAGNOSTIC_OUTPUT_ROOT)

window_means, registries, failures = build_window_means(
    CASES, variables, data_root=INPUT_DATA_ROOT, reference_root=OBSERVATION_ROOT,
    land_support_dir=LAND_SUPPORT_DIR, cache_dir=data_dir, clim_years=CLIM_YEARS,
    clim_exclude_years=CLIM_EXCLUDE_YEARS, min_days=MIN_DAYS, max_members=MAX_MEMBERS,
    force_compute=FORCE_COMPUTE,
)
if failures:
    print(f"\n{len(failures)} request(s) failed and are skipped below:")
    for season, variable, error in failures:
        print(f"  {season} {variable}: {error}")

scores = window_scores(window_means, regions=REGIONS, landmask_file=LAND_SUPPORT_DIR / "landmask_1x1.nc",
                       remove_mean_bias=REMOVE_MEAN_BIAS)
suffix = "_centered" if REMOVE_MEAN_BIAS else ""
table_path = data_dir / f"window_scores{suffix}.csv"
scores.to_csv(table_path, index=False)
print(f"[TABLE] {table_path}")

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


[WRITE] S0 PSL reference
  PSL: 10 members (daily_mean)
[WRITE] S0 PSL CTRL10-S0
  PSL: 10 members (daily_mean)
[WRITE] S0 PSL CAPT10-S0
  PSL: 40 members (daily_mean)
[WRITE] S0 PSL DART40-S0
[WRITE] S0 TREFHT reference
  TREFHT: 10 members (daily_mean)
[WRITE] S0 TREFHT CTRL10-S0
  TREFHT: 10 members (daily_mean)
[WRITE] S0 TREFHT CAPT10-S0


## 4. Plot and show

One figure per region and plot set (`PLOT`, chosen from `PLOT_SETS`; every set reads the same score table, so changing the selection needs no recomputation): anomaly PCC, MSSS vs climatology, and CRPSS vs the climatological ensemble; each cell's four triangles are the lead windows.

In [ ]:
from util.forecast_window_scorecard import plot_window_scorecard

unknown = set(PLOT) - set(PLOT_SETS)
if unknown:
    raise ValueError(f"Unknown plot sets {sorted(unknown)}; choose from {list(PLOT_SETS)}")
for region in REGIONS:
    for name in PLOT:
        fig = plot_window_scorecard(
            scores, region=region, cases=CASES, variables=PLOT_SETS[name],
            title=f"Biweekly-window errors vs observations: {name.replace('_', ' ')}, {region}",
        )
        output = figure_dir / f"window_scorecard_{name}_{region}{suffix}.png"
        fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
        print(f"[FIGURE] {output}")
        if not SHOW:
            plt.close(fig)

### Score table

In [ ]:
display(
    scores.pivot_table(index=["region", "component", "variable", "season", "experiment"],
                       columns="window", values=["pcc", "msss", "crpss", "rmse", "rmse_clim", "crps", "crps_clim", "bias"], sort=False).round(3)
)

## Notes: variables and references

| Row | Model | Reference | Climatology |
|---|---|---|---|
| PSL | `ts/daily_mean` (mean of the 00, 06, 12, 18Z states) | ERA5 daily (mean of the 00, 06, 12, 18Z analyses) | ERA5 daily |
| U, V, T, Q, Z at 200, 500, 700, 850, 925 hPa | 00Z state from `ts/daily_00z` (once-daily instantaneous output; day d = 00Z on d + 1) | ERA5 6-hourly at the same 00Z instants | ERA5 00Z |
| TREFHT | `ts/daily_mean` (00Z-24Z mean of the 6-h means) | ERA5 daily | ERA5 monthly |
| PRECT | `ts/daily_mean` | GPM IMERG (about 60S-60N) | IMERG daily |
| FLUT | `ts/daily_mean` (from the EAM h1 daily average) | NOAA interpolated OLR | NOAA OLR daily |
| TS | `ts/daily_mean` (00Z-24Z mean of the 6-h means) | ERA5 skin temperature, daily | ERA5 monthly |
| WSPD10 | EAM U10 (10-m wind speed), `ts/daily_mean` | ERA5 daily mean of the 6-hourly sqrt(u10² + v10²) | ERA5 daily |
| LHFLX, SHFLX | `ts/daily_mean` (upward positive) | ERA5 daily | ERA5 monthly |
| FSNS, FSDS, FLDS | `ts/daily_mean` (net shortwave, downward shortwave, downward longwave) | ERA5 daily | ERA5 monthly |
| TSA (land) | ELM 2-m air temperature | ERA5 TREFHT | ERA5 monthly |
| TG (land) | ELM ground temperature | ERA5 skin temperature TS | ERA5 monthly |
| SOILM_5CM (land) | ELM H2OSOI, thickness-weighted top 5 cm | ESA CCI | ESA CCI daily |
| EFLX_LH_TOT (land) | ELM latent heat flux | ERA5 LHFLX | ERA5 monthly |
| FSH (land) | ELM sensible heat flux | ERA5 SHFLX | ERA5 monthly |

Model daily means are labelled by the day they cover; the 00Z states keep their instant. Runs without `ts/daily_mean` get the same daily means from `ts/6hourly` (`util.model_daily`). The pressure-level rows are therefore instantaneous skill at 00Z, not daily-mean skill. PRECT uses IMERG rather than GPCP (the reference in `01_compute_and_plot_ens_ts`) because the daily climatology needs 2001-2017 and the GPCP daily archive covers only 2010-2015; IMERG covers about 60S-60N, so global PRECT scores refer to that band. Where the reference has gaps (IMERG, ESA CCI), both window means use only the days the reference is valid, and gridpoints with fewer than `MIN_DAYS` such days are left out. Units: PSL hPa, Z m, Q g kg-1, PRECT mm day-1, wind m s-1, fluxes W m-2. Surface net longwave (FLNS) is not scored: the ERA5 daily and 6-hourly FLNS files in the archive are corrupt (values 0-21 W m-2; FLNSC all zero), although the monthly FLNS is fine. ERA5 surface fluxes come from the reanalysis model's short forecasts, not direct observations.

Caches: per season, `reference_<variable>_window_means.nc` (reference and climatology window means, each climatology year's window means, valid days) and `<experiment>_<variable>_window_means.nc` (each member's window means); they record the climatology years, initial date, ensemble size, and (PSL, TREFHT) the daily-mean construction, and are rebuilt when those change. Table: `window_scores.csv` (`window_scores_centered.csv` and `_centered` figures with `REMOVE_MEAN_BIAS`; one row per season, variable, experiment, region, and window, with `rmse`, `rmse_clim`, `msss`, `bias`, `crps`, `crps_clim`, `crpss`, and `pcc`). For ERA5 TREFHT, TS, LHFLX, and SHFLX the climatological ensemble comes from monthly means, so its spread is somewhat too small and CRPSS is conservative there.